# Dogs vs Cats Classification using Transfer Learning (MobileNetV2)
This notebook uses a pre-trained MobileNetV2 model to achieve high accuracy quickly.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
Data_dir = 'dogs-vs-cats/train' # Ensure your dataset is here

Img_size = (224, 224) # MobileNetV2 standard size
Batch_size = 32

## Augmentation for training
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    Data_dir,
    target_size=Img_size,
    batch_size=Batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    Data_dir,
    target_size=Img_size,
    batch_size=Batch_size,
    class_mode='binary',
    subset='validation'
)

In [ ]:
## Build Model using Transfer Learning
base_model = MobileNetV2(input_shape=(*Img_size, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('cnn_model.h5', save_best_only=True, monitor='val_accuracy')
]

print("Starting training with MobileNetV2...")
history = model.fit(
    train_generator, 
    epochs=10, 
    validation_data=val_generator, 
    callbacks=callbacks
)

In [ ]:
model.save('cnn_model.h5')
print("Improved model saved as cnn_model.h5")